In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Re-define the validated LCOE function here (will be consolidated into
# src/lcoe.py once this notebook confirms the sensitivity approach works —
# same workflow pattern as the Load Forecasting project)
def calculate_lcoe(capex, fcr, fixed_om, capacity_factor, variable_om=0, fuel_cost=0):
    aep_net_per_kw = (capacity_factor * 8760) / 1000
    lcoe = (capex * fcr + fixed_om) / aep_net_per_kw + variable_om + fuel_cost
    return lcoe

# Load the default table built and validated in Notebook 1
defaults = pd.read_csv("../data/lcoe_defaults.csv")
print(defaults[["Province", "Technology", "LCOE"]])

  Province   Technology        LCOE
0       ON        Solar   68.974791
1       ON         Wind   47.986061
2       ON  Natural Gas  184.946062
3       NS        Solar  118.242498
4       NS         Wind   47.986061
5       NS  Natural Gas   77.918950
6       AB        Solar   78.828332
7       AB         Wind   49.282982
8       AB  Natural Gas   70.842777


## Sensitivity Analysis — Tornado Charts

Matching the approach from NREL's "Range of LCOE Parameters" slides (the wind economics deck): for each
technology, we vary **one input at a time** across a realistic range while holding all others at their
default value, and record how much the final LCOE moves. Plotting all parameters together, sorted by
how much they swing the result, produces a "tornado" shape — the widest bar at top is the input the
tool's users should pay the most attention to getting right.

**Ranges used** (applied uniformly across technologies, since these represent genuine real-world
uncertainty bands rather than technology-specific quirks):
- **CapEx**: ±15% (construction cost uncertainty)
- **Fixed O&M**: ±30% (O&M costs are historically the hardest input to pin down precisely)
- **Capacity factor**: ±5 percentage points (real site-to-site resource variation)
- **FCR**: ±1 percentage point (financing terms vary by lender/project risk profile)
- **Project life**: 20–30 years (25 default) — *note: our current formula doesn't use project life
  directly (FCR already encodes it), so this one needs a design decision — flagged below*

In [2]:
def capital_recovery_factor(discount_rate, project_life):
    """
    Converts a discount rate + project life into a Capital Recovery Factor (CRF) —
    the standard way to annualize a lump-sum capital cost over a project's lifetime.
    This is what we'll use in place of the flat FCR we validated with in Notebook 1.

    Parameters
    ----------
    discount_rate : float
        Real discount rate, as a decimal (e.g. 0.05 for 5%)
    project_life : int
        Project lifetime in years

    Returns
    -------
    float
        Capital Recovery Factor (equivalent to FCR in our earlier formula)
    """
    r = discount_rate
    n = project_life
    crf = (r * (1 + r) ** n) / ((1 + r) ** n - 1)
    return crf


def calculate_lcoe_v2(capex, discount_rate, project_life, fixed_om, capacity_factor,
                        variable_om=0, fuel_cost=0):
    """
    LCOE calculation with discount rate and project life as independent inputs,
    rather than a single pre-baked FCR. Internally derives the Capital Recovery
    Factor (CRF) and uses it the same way FCR was used in the validated v1 formula.
    """
    crf = capital_recovery_factor(discount_rate, project_life)
    aep_net_per_kw = (capacity_factor * 8760) / 1000
    lcoe = (capex * crf + fixed_om) / aep_net_per_kw + variable_om + fuel_cost
    return lcoe

In [3]:
# We know the implied FCR for each technology from Notebook 1's validation
# (Wind 6.40%, Solar 6.11%, Gas 6.53%). Now we need to find the (discount_rate,
# project_life) pair that reproduces each FCR via CRF, so discount rate and
# project life become genuinely independent, adjustable inputs.

def solve_discount_rate(target_fcr, project_life, low=0.001, high=0.20, tol=1e-8, max_iter=200):
    """Simple bisection search for the discount rate that produces a target CRF."""
    for _ in range(max_iter):
        mid = (low + high) / 2
        val = capital_recovery_factor(mid, project_life)
        if abs(val - target_fcr) < tol:
            return mid
        if val < target_fcr:
            low = mid
        else:
            high = mid
    return mid

# Assuming a standard 20-year project life (common convention for Canadian
# renewable PPAs/IESO contracts) as our starting point
ASSUMED_PROJECT_LIFE = 20

implied_fcrs = {"Solar": 0.0611, "Wind": 0.0640, "Natural Gas": 0.0653}

discount_rates = {}
for tech, fcr in implied_fcrs.items():
    r = solve_discount_rate(fcr, ASSUMED_PROJECT_LIFE)
    discount_rates[tech] = r
    print(f"{tech}: implied discount rate at {ASSUMED_PROJECT_LIFE}-year life = {r:.4f} ({r*100:.2f}%)")

Solar: implied discount rate at 20-year life = 0.0199 (1.99%)
Wind: implied discount rate at 20-year life = 0.0248 (2.48%)
Natural Gas: implied discount rate at 20-year life = 0.0269 (2.69%)


In [4]:
# Re-test with 25-year project life — matches the convention used in the
# NREL wind economics deck's reference projects, and is a common assumption
# for solar/wind asset life more broadly
ASSUMED_PROJECT_LIFE_V2 = 25

discount_rates_v2 = {}
for tech, fcr in implied_fcrs.items():
    r = solve_discount_rate(fcr, ASSUMED_PROJECT_LIFE_V2)
    discount_rates_v2[tech] = r
    print(f"{tech}: implied discount rate at {ASSUMED_PROJECT_LIFE_V2}-year life = {r:.4f} ({r*100:.2f}%)")

Solar: implied discount rate at 25-year life = 0.0356 (3.56%)
Wind: implied discount rate at 25-year life = 0.0400 (4.00%)
Natural Gas: implied discount rate at 25-year life = 0.0419 (4.19%)


## Findings — Deriving Discount Rate & Project Life

Testing two project life assumptions to find which one produces believable discount rates when
reverse-solved from IESO's implied FCRs:

- **20-year life** produced implausibly low discount rates (2.0–2.7%) — too low for real infrastructure financing
- **25-year life** produced discount rates of 3.56% (Solar), 4.00% (Wind), 4.19% (Gas) — landing almost
  exactly on NREL's own published real WACC range (3.66–4.01%) for their reference wind projects

**Conclusion: 25 years is the correct project life assumption**, matching both the NREL wind economics
deck's convention and producing financing rates consistent with real published figures. This becomes
our default project life across all technologies, with discount rate now genuinely independent and
technology-specific:

| Technology | Discount Rate (real) | Project Life |
|---|---|---|
| Solar | 3.56% | 25 years |
| Wind | 4.00% | 25 years |
| Natural Gas | 4.19% | 25 years |

**Note on methodology:** our CRF-based approach is a standard simplification of IESO's full FCR
methodology (real FCR also incorporates tax rate and depreciation schedule effects, which pure CRF
does not). The near-exact match to NREL's independently-published WACC figures gives strong confidence
this simplification is sound for the tool's purposes, even though it won't reproduce IESO's number to
the same 0.05 $/MWh precision we achieved with the flat-FCR version in Notebook 1.

In [5]:
# Replace the flat FCR column with independent discount_rate + project_life,
# and recalculate LCOE using the new v2 formula to confirm it still matches
discount_rate_map = {"Solar": 0.0356, "Wind": 0.0400, "Natural Gas": 0.0419}

defaults["DiscountRate"] = defaults["Technology"].map(discount_rate_map)
defaults["ProjectLife"] = 25

defaults["LCOE_v2"] = defaults.apply(
    lambda row: calculate_lcoe_v2(
        capex=row["CapEx"], discount_rate=row["DiscountRate"], project_life=row["ProjectLife"],
        fixed_om=row["FixedOM"], capacity_factor=row["CapacityFactor"],
        variable_om=row["VarOM"], fuel_cost=row["FuelCost"]
    ), axis=1
)

# Sanity check: should closely match the original LCOE column from Notebook 1
print(defaults[["Province", "Technology", "LCOE", "LCOE_v2"]])

  Province   Technology        LCOE     LCOE_v2
0       ON        Solar   68.974791   68.947979
1       ON         Wind   47.986061   47.992616
2       ON  Natural Gas  184.946062  184.952156
3       NS        Solar  118.242498  118.196536
4       NS         Wind   47.986061   47.992616
5       NS  Natural Gas   77.918950   77.920575
6       AB        Solar   78.828332   78.797691
7       AB         Wind   49.282982   49.289714
8       AB  Natural Gas   70.842777   70.844107
